# Modeling - ATP Tennis Match Predictor

This notebook trains and compares models to predict ATP match winners, using the feature set built in `02_feature_engineering.ipynb`. Following the approach: establish a naive baseline first, then increase model complexity (Logistic Regression -> Random Forest -> XGBoost), evaluated on a time-based train/test split so the model is only ever tested on matches that happened after its training data.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score)
from xgboost import XGBClassifier

df = pd.read_csv('data/processed/atp_matches_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 34)

### Selecting features and target

The model needs a clean numeric matrix (X) and the target it's predicting (y). Only the engineered difference features and surface dummies go in, not raw identity columns like player names, which carry no generalisable signal.

In [5]:
surface_cols = [c for c in df.columns if c.startswith('Surface_')]

feature_cols = ['rank_diff', 'points_diff', 'points_data_missing',
                'career_win_rate_diff', 'surface_win_rate_diff',
                'recent_form_diff'] + surface_cols

X = df[feature_cols]
y = df['player_1_won']

print(X.shape, y.shape)
X.head()


(68274, 9) (68274,)


,rank_diff,points_diff,points_data_missing,career_win_rate_diff,surface_win_rate_diff,recent_form_diff,Surface_Clay,Surface_Grass,Surface_Hard
0,47,0,1,0.0,0.0,0.0,False,False,True
1,42,0,1,0.0,0.0,0.0,False,False,True
2,-8,0,1,-0.5,-0.5,-0.5,False,False,True
3,49,0,1,0.0,0.0,0.0,False,False,True
4,-171,0,1,0.0,0.0,0.0,False,False,True


### Time-based train/test split

Matches are split chronologically: train on earlier matches, test on the most recent ones, rather than a random split. This mirrors reality, a real prediction only ever has access to the past, never future results, so testing must respect that same constraint.

In [7]:
df_sorted = df.sort_values('Date').reset_index(drop=True)
X = df_sorted[feature_cols]
y = df_sorted['player_1_won']

split_date = df_sorted['Date'].quantile(0.8, interpolation='nearest')
train_mask = df_sorted['Date'] < split_date
test_mask = df_sorted['Date'] >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {X_train.shape[0]} matches ({df_sorted['Date'][train_mask].min().date()} to {df_sorted['Date'][train_mask].max().date()})")
print(f"Test: {X_test.shape[0]} matches ({df_sorted['Date'][test_mask].min().date()} to {df_sorted['Date'][test_mask].max().date()})")


Train: 54612 matches (2000-01-03 to 2021-03-28)
Test: 13662 matches (2021-03-29 to 2026-07-19)
